In [2]:
import re
import time
import json
import requests
import pandas as pd
from datetime import datetime, timezone

# -----------------------------
# CONFIG
# -----------------------------
SEARCH_TERM = "inmigracion"   # tal cual tu captura
TEMA = "inmigracion"
ETIQUETA = "VERDADERO"            
EMAIL = ""                   # recomendado por OpenAlex: "tuemail@dominio.com" (puede ir vacío)
MAX_WORKS = 500              # cuántos resultados máximo quieres traer
SLEEP_SECS = 0.25            # para ser educados con la API
YEAR_FROM = "2020-01-01"     # opcional
YEAR_TO   = "2026-12-31"     # opcional

# Si no quieres filtrar por años, pon YEAR_FROM = None, YEAR_TO = None
USE_YEAR_FILTER = True

# Para intentar texto completo desde PDF (si existe)
TRY_PDF_TEXT = True

# -----------------------------
# HELPERS
# -----------------------------
def abstract_from_inverted_index(inv_idx: dict) -> str:
    """
    OpenAlex devuelve el abstract como 'abstract_inverted_index' (dict de palabra -> [posiciones]).
    Reconstruimos el texto.
    """
    if not inv_idx:
        return ""
    # posiciones -> palabra
    pos_to_word = {}
    for word, positions in inv_idx.items():
        for p in positions:
            pos_to_word[p] = word
    return " ".join(pos_to_word[p] for p in sorted(pos_to_word.keys()))

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip())

def safe_get(d, path, default=None):
    cur = d
    for key in path:
        if cur is None:
            return default
        if isinstance(key, int):
            if isinstance(cur, list) and len(cur) > key:
                cur = cur[key]
            else:
                return default
        else:
            if isinstance(cur, dict) and key in cur:
                cur = cur[key]
            else:
                return default
    return cur

def extract_text_from_pdf_url(pdf_url: str) -> str:
    """
    Intenta descargar el PDF y extraer texto.
    Requiere: pip install pdfminer.six
    """
    if not pdf_url:
        return ""

    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text

        r = requests.get(pdf_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        pdf_bytes = BytesIO(r.content)
        text = extract_text(pdf_bytes)
        return normalize_whitespace(text)
    except Exception:
        return ""

def openalex_headers():
    h = {"User-Agent": "Mozilla/5.0"}
    # OpenAlex recomienda meter mailto
    if EMAIL:
        h["From"] = EMAIL
    return h

# -----------------------------
# QUERY OPENALEX
# -----------------------------
BASE_URL = "https://api.openalex.org/works"

filters = [
    "is_oa:true",
    "language:es",
    "authorships.institutions.country_code:ES",
]
if USE_YEAR_FILTER and YEAR_FROM and YEAR_TO:
    filters.append(f"from_publication_date:{YEAR_FROM}")
    filters.append(f"to_publication_date:{YEAR_TO}")

params = {
    "search": SEARCH_TERM,
    "filter": ",".join(filters),
    "per_page": 200,
    "cursor": "*",
}

rows = []
extraction_date = datetime.now(timezone.utc).date().isoformat()

fetched = 0
session = requests.Session()
session.headers.update(openalex_headers())

while fetched < MAX_WORKS:
    resp = session.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    results = data.get("results", [])
    if not results:
        break

    for w in results:
        if fetched >= MAX_WORKS:
            break

        title = normalize_whitespace(w.get("title", ""))

        # Cuerpo: intentamos PDF texto (si procede) y si no, abstract reconstruido
        cuerpo = ""
        if TRY_PDF_TEXT:
            pdf_url = safe_get(w, ["primary_location", "pdf_url"]) or safe_get(w, ["open_access", "oa_url"])
            # Ojo: oa_url a veces es landing page, no PDF. primary_location.pdf_url es mejor si existe.
            pdf_text = extract_text_from_pdf_url(pdf_url) if (pdf_url and pdf_url.lower().endswith(".pdf")) else ""
            cuerpo = pdf_text

        if not cuerpo:
            inv = w.get("abstract_inverted_index")
            cuerpo = abstract_from_inverted_index(inv)
            cuerpo = normalize_whitespace(cuerpo)

        # URL: preferimos landing page; si no, DOI; si no, OpenAlex URL
        landing = safe_get(w, ["primary_location", "landing_page_url"])
        doi = w.get("doi")
        url = landing or doi or w.get("id", "")

        # Fecha publicación
        fecha = w.get("publication_date") or ""

        # Fuente: revista/venue si existe
        fuente = safe_get(w, ["primary_location", "source", "display_name"]) \
                 or safe_get(w, ["host_venue", "display_name"]) \
                 or "OpenAlex"

        rows.append({
            "Titular": title,
            "Etiqueta": ETIQUETA,
            "Cuerpo": cuerpo,
            "Tema": TEMA,
            "URL": url,
            "Fecha": fecha,
            "Fuente": fuente,
            "Fecha_extraccion": extraction_date,
        })

        fetched += 1

    # paginación por cursor
    next_cursor = safe_get(data, ["meta", "next_cursor"])
    if not next_cursor:
        break
    params["cursor"] = next_cursor

    time.sleep(SLEEP_SECS)

df = pd.DataFrame(rows, columns=[
    "Titular", "Etiqueta", "Cuerpo", "Tema", "URL", "Fecha", "Fuente", "Fecha_extraccion"
])

# Guardado
out_csv = "openalex_inmigracion_es_oa_ES_2024_2026.csv" if USE_YEAR_FILTER else "openalex_inmigracion_es_oa_ES.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"OK. Filas: {len(df)}. Guardado en: {out_csv}")
print(df.head(3).to_string(index=False))


OK. Filas: 80. Guardado en: openalex_inmigracion_es_oa_ES_2024_2026.csv
                                                                           Titular  Etiqueta                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [5]:
import os
import re
import time
import requests
import pandas as pd
from datetime import datetime, timezone

# -------------------------
# CONFIG
# -------------------------

SERPAPI_KEY = os.getenv("da083bfd45a7f5141ec9e5315ec794ac43cdce82d72a77b8faf1024210477eb3") 
TEMA = "inmigracion"
ETIQUETA = "VERDADERO"

YEAR_FROM = 2020
YEAR_TO = 2026

# Aproximaciones a tus filtros "Spain" + "Spanish"
# - allintitle: fuerza el término en el título (más parecido a "Title & Abstract includes", pero más estricto)
# - site:.es y (España OR Spain) aproximan "Country=Spain"
BASE_QUERY = 'allintitle: inmigracion (España OR Spain) site:.es'

HL = "es"           # idioma interfaz :contentReference[oaicite:4]{index=4}
LR = "lang_es"      # limitar a idioma español :contentReference[oaicite:5]{index=5}

ONLY_PDF = True     # aproximación a "open access" quedándonos con recursos PDF :contentReference[oaicite:6]{index=6}

MAX_RESULTS = 200   # total a recolectar
SLEEP_SECS = 0.25

# -------------------------
# HELPERS
# -------------------------
def extract_year(text: str):
    if not text:
        return ""
    m = re.search(r"\b(19|20)\d{2}\b", text)
    return m.group(0) if m else ""

def first_pdf_link(result: dict):
    for r in result.get("resources", []) or []:
        if (r.get("file_format") or "").upper() == "PDF" and r.get("link"):
            return r["link"]
    return ""

# -------------------------
# FETCH
# -------------------------
rows = []
start = 0
extraction_date = datetime.now(timezone.utc).date().isoformat()

while len(rows) < MAX_RESULTS:
    params = {
        "engine": "google_scholar",
        "api_key": SERPAPI_KEY,
        "q": BASE_QUERY,      # :contentReference[oaicite:7]{index=7}
        "hl": HL,             # :contentReference[oaicite:8]{index=8}
        "lr": LR,             # :contentReference[oaicite:9]{index=9}
        "as_ylo": YEAR_FROM,  # :contentReference[oaicite:10]{index=10}
        "as_yhi": YEAR_TO,    # :contentReference[oaicite:11]{index=11}
        "as_sdt": "0",        # excluye patentes (default) :contentReference[oaicite:12]{index=12}
        "start": start,       # :contentReference[oaicite:13]{index=13}
        "num": 20,            # máx 20 :contentReference[oaicite:14]{index=14}
    }

    resp = requests.get("https://serpapi.com/search.json", params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    results = data.get("organic_results", []) or []  # :contentReference[oaicite:15]{index=15}
    if not results:
        break

    for r in results:
        title = (r.get("title") or "").strip()
        link = r.get("link") or ""
        snippet = (r.get("snippet") or "").strip()    # :contentReference[oaicite:16]{index=16}
        pub_summary = (r.get("publication_info", {}) or {}).get("summary", "")  # :contentReference[oaicite:17]{index=17}
        year = extract_year(pub_summary)

        pdf = first_pdf_link(r)  # :contentReference[oaicite:18]{index=18}
        if ONLY_PDF and not pdf:
            continue

        url = pdf or link

        # "Fuente": intentamos sacar el dominio/venue del summary (lo más parecido que da Scholar)
        fuente = pub_summary.split(" - ")[-1].strip() if pub_summary else "Google Scholar"

        rows.append({
            "Titular": title,
            "Etiqueta": ETIQUETA,
            "Cuerpo": snippet,          # Scholar normalmente no da abstract completo: usamos snippet
            "Tema": TEMA,
            "URL": url,
            "Fecha": year,              # al menos el año (a menudo Scholar no da fecha exacta)
            "Fuente": fuente,
            "Fecha_extraccion": extraction_date
        })

        if len(rows) >= MAX_RESULTS:
            break

    start += 20
    time.sleep(SLEEP_SECS)

df = pd.DataFrame(rows, columns=[
    "Titular", "Etiqueta", "Cuerpo", "Tema", "URL", "Fecha", "Fuente", "Fecha_extraccion"
])

out_csv = f"scholar_{TEMA}_{YEAR_FROM}_{YEAR_TO}.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"OK: {len(df)} filas -> {out_csv}")


HTTPError: 401 Client Error: Unauthorized for url: https://serpapi.com/search.json?engine=google_scholar&q=allintitle%3A+inmigracion+%28Espa%C3%B1a+OR+Spain%29+site%3A.es&hl=es&lr=lang_es&as_ylo=2020&as_yhi=2026&as_sdt=0&start=0&num=20

In [6]:
k = os.getenv("da083bfd45a7f5141ec9e5315ec794ac43cdce82d72a77b8faf1024210477eb3")
print("Existe SERPAPI_KEY:", k is not None)
print("Longitud:", len(k) if k else 0)
print("Empieza por:", k[:4] if k else None)

Existe SERPAPI_KEY: False
Longitud: 0
Empieza por: None
